# Experiment 5 — Block Cipher Modes of Operation
ECB · CBC · CFB · OFB · CTR using AES-128. Compares ciphertext patterns.

## AES-128 Core (inlined)

In [ ]:
# ── AES constants ──────────────────────────────────────────────
SBOX=[0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
      0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
      0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
      0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
      0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
      0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
      0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
      0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
      0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
      0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
      0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
      0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
      0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
      0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
      0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
      0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16]
INV_SBOX=[0]*256
for _i,_v in enumerate(SBOX): INV_SBOX[_v]=_i
RCON=[0x01,0x02,0x04,0x08,0x10,0x20,0x40,0x80,0x1b,0x36]

def _gf(a,b):
    p=0
    for _ in range(8):
        if b&1: p^=a
        hi=a&0x80; a=(a<<1)&0xff
        if hi: a^=0x1b
        b>>=1
    return p

def _b2s(b): return [[b[r+4*c] for c in range(4)] for r in range(4)]
def _s2b(s): return bytes(s[r][c] for c in range(4) for r in range(4))
def _sb(s):  return [[SBOX[s[r][c]] for c in range(4)] for r in range(4)]
def _isb(s): return [[INV_SBOX[s[r][c]] for c in range(4)] for r in range(4)]
def _sr(s):  return [[(s[r][(c+r)%4]) for c in range(4)] for r in range(4)]
def _isr(s): return [[(s[r][(c-r)%4]) for c in range(4)] for r in range(4)]
def _ark(s,rk): return [[s[r][c]^rk[r][c] for c in range(4)] for r in range(4)]

def _mc(s):
    o=[[0]*4 for _ in range(4)]
    for c in range(4):
        col=[s[r][c] for r in range(4)]
        o[0][c]=_gf(2,col[0])^_gf(3,col[1])^col[2]^col[3]
        o[1][c]=col[0]^_gf(2,col[1])^_gf(3,col[2])^col[3]
        o[2][c]=col[0]^col[1]^_gf(2,col[2])^_gf(3,col[3])
        o[3][c]=_gf(3,col[0])^col[1]^col[2]^_gf(2,col[3])
    return o

def _imc(s):
    o=[[0]*4 for _ in range(4)]
    for c in range(4):
        col=[s[r][c] for r in range(4)]
        o[0][c]=_gf(0xe,col[0])^_gf(0xb,col[1])^_gf(0xd,col[2])^_gf(0x9,col[3])
        o[1][c]=_gf(0x9,col[0])^_gf(0xe,col[1])^_gf(0xb,col[2])^_gf(0xd,col[3])
        o[2][c]=_gf(0xd,col[0])^_gf(0x9,col[1])^_gf(0xe,col[2])^_gf(0xb,col[3])
        o[3][c]=_gf(0xb,col[0])^_gf(0xd,col[1])^_gf(0x9,col[2])^_gf(0xe,col[3])
    return o

def _kexp(key):
    w=list(key)
    for i in range(4,44):
        t=w[(i-1)*4:i*4]
        if i%4==0:
            t=[SBOX[t[1]],SBOX[t[2]],SBOX[t[3]],SBOX[t[0]]]; t[0]^=RCON[i//4-1]
        w+=[w[(i-4)*4+j]^t[j] for j in range(4)]
    return [_b2s(bytes(w[r*16:(r+1)*16])) for r in range(11)]

def _aes_enc_blk(b,key):
    rk=_kexp(key); s=_ark(_b2s(b),rk[0])
    for r in range(1,11):
        s=_sb(s); s=_sr(s)
        if r<10: s=_mc(s)
        s=_ark(s,rk[r])
    return _s2b(s)

def _aes_dec_blk(b,key):
    rk=_kexp(key); s=_ark(_b2s(b),rk[10])
    for r in range(9,-1,-1):
        s=_isr(s); s=_isb(s); s=_ark(s,rk[r])
        if r>0: s=_imc(s)
    return _s2b(s)

def _pad(d,bs=16): p=bs-(len(d)%bs); return d+bytes([p]*p)
def _unpad(d): return d[:-d[-1]]
def _xb(a,b): return bytes(x^y for x,y in zip(a,b))
BS=16
print("AES-128 core loaded ✓")


## ECB Mode

In [ ]:
def ecb_enc(pt,key):
    p=_pad(pt)
    return b''.join(_aes_enc_blk(p[i:i+BS],key) for i in range(0,len(p),BS))
def ecb_dec(ct,key):
    return _unpad(b''.join(_aes_dec_blk(ct[i:i+BS],key) for i in range(0,len(ct),BS)))


## CBC Mode

In [ ]:
def cbc_enc(pt,key,iv):
    p=_pad(pt); prev=iv; ct=b''
    for i in range(0,len(p),BS):
        blk=_aes_enc_blk(_xb(p[i:i+BS],prev),key); ct+=blk; prev=blk
    return ct
def cbc_dec(ct,key,iv):
    prev=iv; pt=b''
    for i in range(0,len(ct),BS):
        pt+=_xb(_aes_dec_blk(ct[i:i+BS],key),prev); prev=ct[i:i+BS]
    return _unpad(pt)


## CFB Mode

In [ ]:
def cfb_enc(pt,key,iv):
    p=_pad(pt); prev=iv; ct=b''
    for i in range(0,len(p),BS):
        blk=_xb(p[i:i+BS],_aes_enc_blk(prev,key)); ct+=blk; prev=blk
    return ct
def cfb_dec(ct,key,iv):
    prev=iv; pt=b''
    for i in range(0,len(ct),BS):
        pt+=_xb(ct[i:i+BS],_aes_enc_blk(prev,key)); prev=ct[i:i+BS]
    return _unpad(pt)


## OFB Mode

In [ ]:
def ofb_enc(pt,key,iv):
    p=_pad(pt); ks=b''; s=iv
    while len(ks)<len(p): s=_aes_enc_blk(s,key); ks+=s
    return bytes(a^b for a,b in zip(p,ks))
def ofb_dec(ct,key,iv):
    ks=b''; s=iv
    while len(ks)<len(ct): s=_aes_enc_blk(s,key); ks+=s
    return _unpad(bytes(a^b for a,b in zip(ct,ks)))


## CTR Mode

In [ ]:
def ctr_enc(pt,key,nonce):
    ks=b''; ctr=0
    while len(ks)<len(pt):
        ks+=_aes_enc_blk(nonce+ctr.to_bytes(8,'big'),key); ctr+=1
    return bytes(a^b for a,b in zip(pt,ks))
def ctr_dec(ct,key,nonce): return ctr_enc(ct,key,nonce)


## Pattern Comparison — All 5 Modes

In [ ]:
key   = b'0123456789ABCDEF'
iv    = b'\x00'*16
nonce = b'\x00'*8
# Two identical blocks → ECB leaks patterns
pt = b'AAAAAAAAAAAAAAAA'*2

tests = [
    ("ECB", ecb_enc(pt,key),      lambda c: ecb_dec(c,key)),
    ("CBC", cbc_enc(pt,key,iv),   lambda c: cbc_dec(c,key,iv)),
    ("CFB", cfb_enc(pt,key,iv),   lambda c: cfb_dec(c,key,iv)),
    ("OFB", ofb_enc(pt,key,iv),   lambda c: ofb_dec(c,key,iv)),
    ("CTR", ctr_enc(pt,key,nonce),lambda c: ctr_dec(c,key,nonce)),
]

print(f"{'Mode':<5} | {'CT (first 32 hex)':<32} | Block1==Block2?")
print('-'*70)
for mode,ct,dec_fn in tests:
    b1=ct[:BS].hex().upper(); b2=ct[BS:2*BS].hex().upper()
    same="SAME ⚠ ECB LEAK!" if b1==b2 else "Different ✓"
    print(f"{mode:<5} | {b1} | {same}")
    assert dec_fn(ct)==pt, f"{mode} decrypt failed"

print()
print("✓ All 5 modes encrypt/decrypt correctly")
print()
print("ECB: identical plaintext blocks → identical ciphertext blocks (pattern leak!)")
print("CBC/CFB/OFB/CTR: chaining/feedback → no visible patterns")


## ECB vs CBC Pattern Leakage — Structured Data

In [ ]:
structured = b'ATTACK AT DAWN!!' * 3
ecb_ct = ecb_enc(structured, key)
cbc_ct = cbc_enc(structured, key, iv)
ecb_blks = [ecb_ct[i:i+BS].hex() for i in range(0,len(ecb_ct)-BS,BS)]
cbc_blks = [cbc_ct[i:i+BS].hex() for i in range(0,len(cbc_ct)-BS,BS)]
print(f"3 identical plaintext blocks:")
print(f"  ECB unique ciphertext blocks: {len(set(ecb_blks))}/3  ← patterns visible")
print(f"  CBC unique ciphertext blocks: {len(set(cbc_blks))}/3  ← no patterns")
